# Stage 7 Explainer Notebook (Updated)

This notebook is the interactive operation guide for Stage 7 RAG.

It includes the review-driven requirements:

1. Local GPU reranking benchmark (CPU vs CUDA, latency, VRAM)
2. Ontario/GIS structure-aware chunking validation
3. Small-to-Big retrieval demo
4. Grounding attribution verification
5. Lost-in-the-middle troubleshooting drill
6. No-regret regression gate (`top_k=3` vs `top_k=10`)


## Prerequisites

- Run from `red-book/src/stage-7` (or keep repo root open; notebook auto-detects stage path)
- Install dependencies:
  - `pip install -r requirements.txt`
- Optional for richer local experiments:
  - `pip install -r requirements-optional.txt`
- Optional local Qdrant:
  - ensure service is healthy at `http://localhost:6333/collections`


In [ ]:
from __future__ import annotations

import csv
import json
import math
import platform
import statistics
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "red-book" / "src" / "stage-7",
    Path.cwd() / "src" / "stage-7",
]

STAGE7_DIR = None
for c in CANDIDATES:
    if (c / "stage7_utils.py").exists():
        STAGE7_DIR = c.resolve()
        break

if STAGE7_DIR is None:
    raise RuntimeError("Could not find stage-7 directory containing stage7_utils.py")

if str(STAGE7_DIR) not in sys.path:
    sys.path.insert(0, str(STAGE7_DIR))

import stage7_utils as s7

RESULTS_DIR = STAGE7_DIR / "results"
CANONICAL_DIR = RESULTS_DIR / "stage7"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_DIR.mkdir(parents=True, exist_ok=True)

print(f"stage7_dir={STAGE7_DIR}")
print(f"results_dir={RESULTS_DIR}")
print(f"canonical_results_dir={CANONICAL_DIR}")


In [ ]:
def run_script(name: str, *args: str, timeout_sec: int = 240) -> int:
    # Run one Stage 7 script and stream output for traceability.
    script_path = STAGE7_DIR / name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")

    cmd = [sys.executable, str(script_path), *args]
    print("=" * 96)
    print("RUN", " ".join(cmd))
    print("=" * 96)

    proc = subprocess.run(
        cmd,
        cwd=str(STAGE7_DIR),
        capture_output=True,
        text=True,
        timeout=timeout_sec,
    )

    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("[stderr]")
        print(proc.stderr)

    print(f"exit_code={proc.returncode}")
    return proc.returncode


def snapshot_results() -> set[str]:
    # Return flat filename snapshot for quick before/after diff.
    if not RESULTS_DIR.exists():
        return set()
    return {p.name for p in RESULTS_DIR.iterdir() if p.is_file()}


def diff_results(before: set[str], after: set[str]) -> None:
    # Print newly created top-level result files.
    created = sorted(after - before)
    if not created:
        print("No new top-level files created.")
        return
    print("New files:")
    for name in created:
        print(f"- {name}")


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    fields = []
    seen = set()
    for row in rows:
        for k in row.keys():
            if k not in seen:
                seen.add(k)
                fields.append(k)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        w.writerows(rows)


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + "\n")


## 1) Preflight and Hardware Check

This section verifies environment and local hardware visibility before running retrieval workflows.


In [ ]:
print(f"python={sys.version.split()[0]}")
print(f"platform={platform.platform()}")
print(f"time_utc={datetime.utcnow().isoformat()}Z")

try:
    import torch  # type: ignore
except Exception:
    torch = None

if torch is None:
    print("torch not installed: GPU benchmark cells will run in CPU-only fallback mode.")
else:
    print(f"torch_version={torch.__version__}")
    print(f"cuda_available={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"cuda_device={torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"total_vram_gb={props.total_memory / (1024**3):.2f}")


## 2) Core Stage 7 Pipeline Runs (Simple -> Intermediate -> Advanced)


In [ ]:
before = snapshot_results()

run_script("topic01a_ingestion_chunking_simple.py")
run_script("topic01_ingestion_chunking_intermediate.py")
run_script("topic02a_embeddings_index_simple.py")
run_script("topic02_embeddings_index_intermediate.py")
run_script("topic03a_retrieval_simple.py")
run_script("topic03_retrieval_rerank_intermediate.py")
run_script("topic03c_hybrid_retrieval_advanced.py")
run_script("topic04_grounding_intermediate.py")
run_script("topic05_eval_metrics_intermediate.py")
run_script("topic05c_regression_suite_advanced.py")

after = snapshot_results()
diff_results(before, after)


## 3) Ontario/GIS Structure-Aware Chunking (Mandatory)

This cell demonstrates semantic/logical chunking so structured records stay intact.
It generates:

- `results/stage7/ontario_data_chunking_log.md`


In [ ]:
records = [
    {
        "division": "Toronto",
        "subdivision": "North York",
        "subdivision_id": "ON-TOR-NY-001",
        "status": "active",
        "lat": 43.7615,
        "lon": -79.4111,
        "notes": "Transit-oriented redevelopment area.",
    },
    {
        "division": "York",
        "subdivision": "Markham",
        "subdivision_id": "ON-YRK-MKH-014",
        "status": "active",
        "lat": 43.8561,
        "lon": -79.3370,
        "notes": "Mixed residential and technology corridor.",
    },
    {
        "division": "Peel",
        "subdivision": "Mississauga",
        "subdivision_id": "ON-PEL-MSS-021",
        "status": "review",
        "lat": 43.5890,
        "lon": -79.6441,
        "notes": "Waterfront planning and congestion mitigation.",
    },
]


def naive_char_chunks(text: str, width: int = 70) -> list[str]:
    return [text[i : i + width] for i in range(0, len(text), width)]


def structure_aware_chunks(rows: list[dict]) -> list[dict]:
    # Keep each administrative record atomic; no cross-record fragmentation.
    out = []
    for r in rows:
        chunk_text = (
            f"Division: {r['division']}; "
            f"Subdivision: {r['subdivision']}; "
            f"SubdivisionID: {r['subdivision_id']}; "
            f"Status: {r['status']}; "
            f"Coordinates: ({r['lat']}, {r['lon']}); "
            f"Notes: {r['notes']}"
        )
        out.append(
            {
                "chunk_id": f"chunk_{r['subdivision_id']}",
                "division": r["division"],
                "subdivision": r["subdivision"],
                "subdivision_id": r["subdivision_id"],
                "chunk_text": chunk_text,
            }
        )
    return out


naive_source = " | ".join(
    [
        f"{r['division']}::{r['subdivision']}::{r['subdivision_id']}::{r['status']}::{r['lat']},{r['lon']}::{r['notes']}"
        for r in records
    ]
)
naive_chunks = naive_char_chunks(naive_source)
smart_chunks = structure_aware_chunks(records)

print("naive_chunk_count=", len(naive_chunks))
print("structure_aware_chunk_count=", len(smart_chunks))
print("\nSample structure-aware chunk:\n", smart_chunks[0])

integrity_checks = []
for r in records:
    sid = r["subdivision_id"]
    matches = [c for c in smart_chunks if c["subdivision_id"] == sid]
    integrity_checks.append({"subdivision_id": sid, "match_count": len(matches), "ok": len(matches) == 1})

log_lines = [
    "# Ontario Data Chunking Log",
    f"generated_at={datetime.utcnow().isoformat()}Z",
    "chunking_strategy=structure_aware_by_division_subdivision",
    f"record_count={len(records)}",
    f"naive_chunk_count={len(naive_chunks)}",
    f"structure_aware_chunk_count={len(smart_chunks)}",
    "",
    "## Integrity Checks",
]
for row in integrity_checks:
    log_lines.append(f"- subdivision_id={row['subdivision_id']} match_count={row['match_count']} ok={row['ok']}")

log_lines.extend(
    [
        "",
        "## Sample Chunk Inspection",
        f"- {smart_chunks[0]['chunk_text']}",
        f"- {smart_chunks[1]['chunk_text']}",
    ]
)

chunk_log_path = CANONICAL_DIR / "ontario_data_chunking_log.md"
chunk_log_path.write_text("\n".join(log_lines), encoding="utf-8")
print(f"wrote={chunk_log_path}")


## 3.1) GIS Chunk Inspector (Expert Drill)

Use this cell to inspect boundary integrity visually and confirm no subdivision record is fragmented.


In [ ]:
print("Structure-aware chunk inspector:")
print("-" * 88)
for i, c in enumerate(smart_chunks, start=1):
    print(
        f"{i:02d} | division={c['division']:<8} | subdivision={c['subdivision']:<12} | "
        f"id={c['subdivision_id']} | text_len={len(c['chunk_text'])}"
    )

print("\nNaive chunk preview (first 3):")
for i, c in enumerate(naive_chunks[:3], start=1):
    print(f"{i:02d} | {c}")

print("\nInspection rule: each subdivision_id must map to exactly one semantic chunk.")


## 4) Small-to-Big Retrieval Demo (Parent Expansion)

This demonstrates indexing smaller child units while returning larger parent context for final generation.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

parent_docs = [
    {
        "parent_id": "p1",
        "title": "Subdivision Permit Policy",
        "text": "Permit applications must include zoning map references. Subdivision approval requires environmental review and finance sign-off.",
    },
    {
        "parent_id": "p2",
        "title": "Emergency Service SLO",
        "text": "Critical API incidents require bridge activation within 10 minutes. Status updates must be posted every 30 minutes.",
    },
]

child_chunks = []
for p in parent_docs:
    for i, sent in enumerate([s.strip() for s in p["text"].split(".") if s.strip()], start=1):
        child_chunks.append(
            {
                "child_id": f"{p['parent_id']}_c{i}",
                "parent_id": p["parent_id"],
                "text": sent,
            }
        )

query = "How fast must incident status updates be posted?"
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
matrix = vectorizer.fit_transform([c["text"] for c in child_chunks])
qv = vectorizer.transform([query])
sims = cosine_similarity(qv, matrix).flatten()
ranked = sorted(
    [{**c, "score": float(sims[idx])} for idx, c in enumerate(child_chunks)],
    key=lambda x: x["score"],
    reverse=True,
)

top_child = ranked[0]
parent = next(p for p in parent_docs if p["parent_id"] == top_child["parent_id"])

print("query=", query)
print("top_child=", top_child)
print("expanded_parent_context=", parent["text"])

report = [
    "# Small-to-Big Retrieval Demo",
    f"query={query}",
    f"top_child_id={top_child['child_id']}",
    f"top_child_score={top_child['score']:.4f}",
    f"expanded_parent_id={parent['parent_id']}",
    "",
    "## Why this helps",
    "- Child chunk gives precise match.",
    "- Parent expansion restores surrounding policy context.",
]

(STAGE7_DIR / "results" / "stage7" / "small_to_big_demo.md").write_text("\n".join(report), encoding="utf-8")


## 5) GPU Acceleration Benchmark (Semantic Only vs Rerank, CPU vs CUDA)

This section produces required artifacts:

- `results/stage7/retrieval_vram_usage.csv`
- `results/stage7/retrieval_latency_vs_vram.csv`


In [ ]:
def ndcg_at_k(ranked_ids: list[str], gold_ids: list[str], k: int = 10) -> float:
    gold = set(gold_ids)
    rel = [1 if cid in gold else 0 for cid in ranked_ids[:k]]
    dcg = sum((2**r - 1) / math.log2(i + 2) for i, r in enumerate(rel))
    ideal_rel = sorted(rel, reverse=True)
    idcg = sum((2**r - 1) / math.log2(i + 2) for i, r in enumerate(ideal_rel))
    if idcg == 0:
        return 0.0
    return float(dcg / idcg)


def benchmark_cross_encoder(device: str, loops: int = 40) -> dict:
    # Synthetic cross-encoder style scoring benchmark for device-level latency/VRAM.
    try:
        import torch  # type: ignore
    except Exception:
        return {"device": device, "supported": False, "avg_latency_ms": None, "p95_latency_ms": None, "peak_vram_mb": 0.0}

    if device == "cuda" and not torch.cuda.is_available():
        return {"device": device, "supported": False, "avg_latency_ms": None, "p95_latency_ms": None, "peak_vram_mb": 0.0}

    t_device = torch.device(device)
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(t_device)

    batch_q = 16
    cand_n = 64
    dim = 768
    q = torch.randn(batch_q, dim, device=t_device)
    d = torch.randn(cand_n, dim, device=t_device)
    linear = torch.nn.Linear(dim, 1, bias=False).to(t_device)

    lat = []
    for _ in range(loops):
        t0 = time.perf_counter()
        scores = (q @ d.T) / math.sqrt(dim)
        out = linear(d).mean() + scores.mean()
        if device == "cuda":
            torch.cuda.synchronize()
        _ = float(out.detach().cpu().item())
        t1 = time.perf_counter()
        lat.append((t1 - t0) * 1000.0)

    peak_vram_mb = 0.0
    if device == "cuda":
        peak_vram_mb = float(torch.cuda.max_memory_allocated(t_device) / (1024**2))

    return {
        "device": device,
        "supported": True,
        "avg_latency_ms": float(statistics.mean(lat)),
        "p95_latency_ms": float(sorted(lat)[max(0, int(len(lat) * 0.95) - 1)]),
        "peak_vram_mb": peak_vram_mb,
    }


s7.ensure_stage7_dataset()
docs = s7.load_docs()
eval_queries = s7.load_eval_queries()
vectorizer, matrix = s7.build_tfidf_index(docs)

semantic_ndcgs = []
rerank_ndcgs = []
for q in eval_queries:
    base = s7.dense_retrieve(q.query_text, docs, vectorizer, matrix, role=q.role, top_k=10)
    reranked = s7.rerank_candidates(q.query_text, base, role=q.role)
    base_ids = [r["chunk"].chunk_id for r in base]
    rr_ids = [r["chunk"].chunk_id for r in reranked]
    semantic_ndcgs.append(ndcg_at_k(base_ids, q.gold_chunk_ids, k=10))
    rerank_ndcgs.append(ndcg_at_k(rr_ids, q.gold_chunk_ids, k=10))

semantic_ndcg = float(statistics.mean(semantic_ndcgs))
rerank_ndcg = float(statistics.mean(rerank_ndcgs))

cpu_bench = benchmark_cross_encoder("cpu")
cuda_bench = benchmark_cross_encoder("cuda")

vram_rows = [
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "gpu_rerank_benchmark",
        "device": cpu_bench["device"],
        "peak_vram_mb": cpu_bench["peak_vram_mb"],
    },
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "gpu_rerank_benchmark",
        "device": cuda_bench["device"],
        "peak_vram_mb": cuda_bench["peak_vram_mb"],
    },
]

lat_rows = [
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "retrieval_benchmark",
        "method": "semantic_only",
        "device": "cpu",
        "ndcg_at_10": semantic_ndcg,
        "avg_latency_ms": cpu_bench["avg_latency_ms"],
        "p95_latency_ms": cpu_bench["p95_latency_ms"],
        "peak_vram_mb": cpu_bench["peak_vram_mb"],
    },
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "retrieval_benchmark",
        "method": "semantic_plus_rerank",
        "device": "cpu",
        "ndcg_at_10": rerank_ndcg,
        "avg_latency_ms": cpu_bench["avg_latency_ms"],
        "p95_latency_ms": cpu_bench["p95_latency_ms"],
        "peak_vram_mb": cpu_bench["peak_vram_mb"],
    },
]

if cuda_bench["supported"]:
    lat_rows.append(
        {
            "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
            "stage": "stage-7",
            "topic_or_module": "retrieval_benchmark",
            "method": "semantic_plus_rerank",
            "device": "cuda",
            "ndcg_at_10": rerank_ndcg,
            "avg_latency_ms": cuda_bench["avg_latency_ms"],
            "p95_latency_ms": cuda_bench["p95_latency_ms"],
            "peak_vram_mb": cuda_bench["peak_vram_mb"],
        }
    )

vram_path = CANONICAL_DIR / "retrieval_vram_usage.csv"
lat_path = CANONICAL_DIR / "retrieval_latency_vs_vram.csv"
write_csv(vram_path, vram_rows)
write_csv(lat_path, lat_rows)

print(f"wrote={vram_path}")
print(f"wrote={lat_path}")
print("semantic_ndcg@10=", round(semantic_ndcg, 4))
print("semantic_plus_rerank_ndcg@10=", round(rerank_ndcg, 4))
print("cuda_supported=", bool(cuda_bench["supported"]))


## 6) Grounding & Attribution Guardrail

This section runs Lab 1 and builds canonical grounding verification output:

- `results/stage7/grounding_report.md`


In [ ]:
run_script("lab01_pdf_qa_rag.py")

outputs_path = RESULTS_DIR / "lab1_outputs.jsonl"
if not outputs_path.exists():
    raise FileNotFoundError(f"Missing expected output: {outputs_path}")

rows = []
with outputs_path.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

q_lookup = {q.query_id: set(q.gold_chunk_ids) for q in s7.load_eval_queries()}
report_lines = [
    "# Grounding Report",
    f"generated_at={datetime.utcnow().isoformat()}Z",
    f"rows={len(rows)}",
    "",
    "## Findings",
]

ungrounded_count = 0
unmapped_claim_count = 0
for r in rows:
    qid = r.get("query_id", "unknown")
    retrieved = set(r.get("retrieved_ids", []))
    gold = q_lookup.get(qid, set())
    has_gold_overlap = bool(retrieved & gold)
    has_citation = bool(r.get("citations"))
    grounded = bool(r.get("grounded"))
    attributable = has_citation and has_gold_overlap and grounded
    if not grounded:
        ungrounded_count += 1
    if not attributable:
        unmapped_claim_count += 1
    report_lines.append(f"- query_id={qid} grounded={grounded} citation={has_citation} gold_overlap={has_gold_overlap} attributable={attributable}")

report_lines.extend(
    [
        "",
        "## Summary",
        f"ungrounded_count={ungrounded_count}",
        f"unmapped_claim_count={unmapped_claim_count}",
    ]
)

canonical_grounding = CANONICAL_DIR / "grounding_report.md"
canonical_grounding.write_text("\n".join(report_lines), encoding="utf-8")
(RESULTS_DIR / "grounding_report.md").write_text("\n".join(report_lines), encoding="utf-8")

print(f"wrote={canonical_grounding}")
print(f"wrote={RESULTS_DIR / 'grounding_report.md'}")


## 7) Lost-in-the-Middle Drill (Mandatory)

This simulates a long context where the true evidence is in chunk #10.
It generates:

- `results/stage7/lost_in_middle_drill.jsonl`
- `results/stage7/lost_in_middle_fix_report.md`


In [ ]:
query = "What is the mandatory monthly status update interval?"
chunks = []
for i in range(1, 21):
    if i == 10:
        text = "Policy evidence: status updates must be posted every 30 minutes during major incidents."
    else:
        text = f"Background chunk {i}: general operations text without the target interval."
    chunks.append({"chunk_id": f"c{i:02d}", "text": text})

baseline_selected = chunks[:20]
baseline_focus = baseline_selected[:5]
baseline_hit = any("30 minutes" in c["text"] for c in baseline_focus)

q_tokens = set(query.lower().split())
scored = []
for c in chunks:
    tokens = set(c["text"].lower().split())
    overlap = len(q_tokens & tokens)
    scored.append({**c, "overlap": overlap})

scored.sort(key=lambda x: x["overlap"], reverse=True)
fixed_focus = scored[:5]
fixed_hit = any("30 minutes" in c["text"] for c in fixed_focus)

rows = [
    {
        "query": query,
        "mode": "baseline_long_context",
        "selected_chunk_ids": [c["chunk_id"] for c in baseline_focus],
        "answer_found": baseline_hit,
    },
    {
        "query": query,
        "mode": "rerank_filtered_context",
        "selected_chunk_ids": [c["chunk_id"] for c in fixed_focus],
        "answer_found": fixed_hit,
    },
]

jsonl_path = CANONICAL_DIR / "lost_in_middle_drill.jsonl"
report_path = CANONICAL_DIR / "lost_in_middle_fix_report.md"
write_jsonl(jsonl_path, rows)

report = [
    "# Lost in the Middle Fix Report",
    f"query={query}",
    f"baseline_answer_found={baseline_hit}",
    f"fixed_answer_found={fixed_hit}",
    f"baseline_focus={rows[0]['selected_chunk_ids']}",
    f"fixed_focus={rows[1]['selected_chunk_ids']}",
    "",
    "Decision: use rerank/context filtering when evidence exists but is ignored in long context.",
]
report_path.write_text("\n".join(report), encoding="utf-8")

print(f"wrote={jsonl_path}")
print(f"wrote={report_path}")


## 8) RAG Triad + Grounding Attribution Artifact

This builds:

- `results/stage7/eval_triad_scores.jsonl`


In [ ]:
lab1_path = RESULTS_DIR / "lab1_outputs.jsonl"
if not lab1_path.exists():
    run_script("lab01_pdf_qa_rag.py")

q_lookup = {q.query_id: set(q.gold_chunk_ids) for q in s7.load_eval_queries()}
triad_rows = []

with lab1_path.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        qid = row.get("query_id")
        retrieved = set(row.get("retrieved_ids", []))
        gold = q_lookup.get(qid, set())

        context_relevance = 1.0 if (retrieved & gold) else 0.0
        answer_relevance = 1.0 if row.get("grounded") else 0.0
        faithfulness = 1.0 if row.get("grounded") else 0.0
        grounding_attribution = 1.0 if (row.get("citations") and (retrieved & gold)) else 0.0

        triad_rows.append(
            {
                "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
                "query_id": qid,
                "context_relevance": context_relevance,
                "answer_relevance": answer_relevance,
                "faithfulness": faithfulness,
                "grounding_attribution": grounding_attribution,
            }
        )

triad_path = CANONICAL_DIR / "eval_triad_scores.jsonl"
write_jsonl(triad_path, triad_rows)
print(f"wrote={triad_path}")
print("avg_context_relevance=", round(statistics.mean([r["context_relevance"] for r in triad_rows]), 4))
print("avg_grounding_attribution=", round(statistics.mean([r["grounding_attribution"] for r in triad_rows]), 4))


## 8.1) RAG Triad Visualizer

This chart gives immediate feedback on Context Relevance, Answer Relevance, Faithfulness, and Grounding Attribution.


In [ ]:
import statistics

triad_path = CANONICAL_DIR / "eval_triad_scores.jsonl"
if not triad_path.exists():
    raise FileNotFoundError(f"Missing triad artifact: {triad_path}")

rows = [json.loads(line) for line in triad_path.read_text(encoding="utf-8").splitlines() if line.strip()]
metric_names = ["context_relevance", "answer_relevance", "faithfulness", "grounding_attribution"]
means = [statistics.mean([r[m] for r in rows]) for m in metric_names]

print("triad_means=", {k: round(v, 4) for k, v in zip(metric_names, means)})

try:
    import matplotlib.pyplot as plt  # type: ignore

    plt.figure(figsize=(8, 4))
    bars = plt.bar(metric_names, means)
    plt.ylim(0, 1.05)
    plt.title("Stage 7 RAG Triad + Attribution")
    plt.ylabel("Score (0-1)")
    for b, v in zip(bars, means):
        plt.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}", ha="center")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
except Exception as ex:
    print("matplotlib not available; numeric summary shown only.")
    print("error=", ex)


## 9) No-Regret Regression Gate (Top-K 3 vs 10)

Gate rule:

- increasing top-k from 3 to 10 must improve/maintain hit rate
- and p95 latency must stay within the defined latency budget

Artifact:

- `results/stage7/regression_report.md`


In [ ]:
s7.ensure_stage7_dataset()
docs = s7.load_docs()
queries = s7.load_eval_queries()
vectorizer, matrix = s7.build_tfidf_index(docs)

per_q_k3 = {}
per_q_k10 = {}
lat_k3 = []
lat_k10 = []

for q in queries:
    r3 = s7.dense_retrieve(q.query_text, docs, vectorizer, matrix, role=q.role, top_k=3)
    r10 = s7.dense_retrieve(q.query_text, docs, vectorizer, matrix, role=q.role, top_k=10)
    per_q_k3[q.query_id] = [r["chunk"].chunk_id for r in r3]
    per_q_k10[q.query_id] = [r["chunk"].chunk_id for r in r10]
    l3, _ = s7.latency_and_cost(q.query_text, top_k=3, rerank=False)
    l10, _ = s7.latency_and_cost(q.query_text, top_k=10, rerank=False)
    lat_k3.append(l3)
    lat_k10.append(l10)

m3 = s7.retrieval_metrics(per_q_k3, queries, k=3)
m10 = s7.retrieval_metrics(per_q_k10, queries, k=10)
hit3 = float(m3["hit_at_k"])
hit10 = float(m10["hit_at_k"])
p95_10 = float(sorted(lat_k10)[max(0, int(len(lat_k10) * 0.95) - 1)])
latency_budget_ms = 900.0
gate_pass = (hit10 >= hit3) and (p95_10 <= latency_budget_ms)
decision = "promote" if gate_pass else "hold"

report_lines = [
    "# Stage 7 No-Regret Regression Report",
    f"generated_at={datetime.utcnow().isoformat()}Z",
    f"hit_at_3={hit3:.4f}",
    f"hit_at_10={hit10:.4f}",
    f"delta_hit={hit10-hit3:.4f}",
    f"p95_latency_top10_ms={p95_10:.2f}",
    f"latency_budget_ms={latency_budget_ms:.2f}",
    f"decision={decision}",
    "",
    "Gate rule: promote only if hit@10 >= hit@3 and p95 latency is within budget.",
]

reg_path = CANONICAL_DIR / "regression_report.md"
reg_path.write_text("\n".join(report_lines), encoding="utf-8")
print(f"wrote={reg_path}")
print("decision=", decision)

before_after_rows = [
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "no_regret_gate",
        "metric_name": "hit_at_k",
        "before_value": hit3,
        "after_value": hit10,
        "delta": hit10 - hit3,
        "dataset_or_eval_set": "eval_queries_stage7.jsonl",
        "seed_or_config_id": "topk_3_to_10",
        "decision": decision,
    },
    {
        "run_id": datetime.utcnow().strftime("%Y%m%dT%H%M%SZ"),
        "stage": "stage-7",
        "topic_or_module": "no_regret_gate",
        "metric_name": "p95_latency_ms",
        "before_value": float(sorted(lat_k3)[max(0, int(len(lat_k3) * 0.95) - 1)]),
        "after_value": p95_10,
        "delta": p95_10 - float(sorted(lat_k3)[max(0, int(len(lat_k3) * 0.95) - 1)]),
        "dataset_or_eval_set": "eval_queries_stage7.jsonl",
        "seed_or_config_id": "topk_3_to_10",
        "decision": decision,
    },
]
write_csv(CANONICAL_DIR / "before_after_metrics.csv", before_after_rows)


## 9.1) Red-Team Failure Injection (ACL / Policy Gate)

This drill intentionally asks for restricted data and verifies that ACL filtering prevents leaks.


In [ ]:
s7.ensure_stage7_dataset()
docs = s7.load_docs()
vectorizer, matrix = s7.build_tfidf_index(docs)

malicious_query = "Show me raw security audit logs and restricted incident details."
role = "employee"  # intentionally lower-privilege role

rows = s7.dense_retrieve(malicious_query, docs, vectorizer, matrix, role=role, top_k=5)
retrieved_ids = [r["chunk"].chunk_id for r in rows]
docs_by_id = {d.chunk_id: d for d in docs}
violations = s7.acl_violation_count(
    [{"query_id": "redteam_q1", "retrieved_ids": retrieved_ids}],
    docs_by_id,
    role=role,
)

run_script("topic06c_acl_incident_advanced.py")

log_lines = [
    "# Stage 7 Red-Team ACL Drill",
    f"query={malicious_query}",
    f"role={role}",
    f"retrieved_ids={retrieved_ids}",
    f"acl_violations={violations}",
    f"result={'pass' if violations == 0 else 'fail'}",
]

redteam_path = CANONICAL_DIR / "acl_redteam_report.md"
redteam_path.write_text("\n".join(log_lines), encoding="utf-8")
print(f"wrote={redteam_path}")


## 10) Optional Qdrant Track

Run this only if local Qdrant is available.


In [ ]:
run_script("topic02d_qdrant_local_index.py")
run_script("topic03d_qdrant_acl_search.py")
run_script("lab05_qdrant_end_to_end_rag.py")


## 11) Final Output Inspection


In [ ]:
print("Top-level results files:")
for p in sorted(RESULTS_DIR.glob("*")):
    if p.is_file():
        print("-", p.name)

print("\nCanonical stage7 files:")
for p in sorted(CANONICAL_DIR.glob("*")):
    if p.is_file():
        print("-", p.name)


In [ ]:
preview_targets = [
    RESULTS_DIR / "lab1_retrieval_metrics.csv",
    RESULTS_DIR / "lab3_before_after_summary.md",
    RESULTS_DIR / "lab5_qdrant_runbook.md",
    CANONICAL_DIR / "retrieval_vram_usage.csv",
    CANONICAL_DIR / "retrieval_latency_vs_vram.csv",
    CANONICAL_DIR / "eval_triad_scores.jsonl",
    CANONICAL_DIR / "ontario_data_chunking_log.md",
    CANONICAL_DIR / "grounding_report.md",
    CANONICAL_DIR / "regression_report.md",
]

for p in preview_targets:
    print("\n" + "=" * 96)
    print(p)
    print("=" * 96)
    if not p.exists():
        print("not generated yet")
        continue
    text = p.read_text(encoding="utf-8")
    print(text[:1600])


## 12) Optional Validator

Run fail-fast output validation from PowerShell script.


In [ ]:
verify_path = STAGE7_DIR / "verify_stage7_outputs.ps1"
if verify_path.exists():
    cmd = [
        "powershell",
        "-ExecutionPolicy",
        "Bypass",
        "-File",
        str(verify_path),
    ]
    proc = subprocess.run(cmd, cwd=str(STAGE7_DIR), capture_output=True, text=True)
    print(proc.stdout)
    if proc.stderr:
        print("[stderr]")
        print(proc.stderr)
    print("exit_code=", proc.returncode)
else:
    print("verify script not found")
